# 5.3 · K 近邻分类 / K-Nearest Neighbors (KNN)

> **课程定位 / Where this fits**
> 第 3 课，**Part 5 · 监督学习：分类**。
> Lesson 3, **Part 5 · Supervised Classification**.
>
> 5.1/5.2 是**参数模型**：训练时学一组权重，学完就把数据丢掉。KNN 完全相反，是**非参数惰性(lazy)**模型的代表：**根本不训练**，把全部训练数据记下来，预测时现场找最近的几个邻居投票。
> 5.1/5.2 are **parametric**: they learn weights during training, then discard the data. KNN is the opposite — the canonical **non-parametric, lazy** model: it **doesn't train at all**, just memorizes the data and, at predict time, finds the nearest neighbors and lets them vote.
>
> 4.10 讲过 KNN 回归，这一课是分类版。
> 4.10 covered KNN regression; this is the classification version.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{x}$ —— 待预测样本 / a query sample
> - $K$ —— 看几个最近邻 / number of nearest neighbors to consult
> - $\|\mathbf{x}-\mathbf{x}_i\|$ —— 欧氏距离 / Euclidean distance
> - $n,\ d$ —— 训练样本数、特征维数 / number of training samples, number of features

> 💡 **面试相关 / Interview-relevant**
> - "KNN 训练/预测的时间复杂度"（★★★★，训练 O(1)、预测 O(nd)）
> - "为什么 KNN 必须做特征缩放"（★★★★）
> - "K 怎么选 / K 对偏差方差的影响"（★★★★）
> - "维度灾难为什么打击 KNN"（★★★★）
> - "KD-Tree / Ball-Tree 加速原理"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解惰性学习：训练 O(1)，代价全在预测。
   Understand lazy learning: O(1) training, all cost at prediction.
2. **从零**实现 KNN 分类（欧氏距离 + 多数投票）。
   Implement KNN **from scratch** (Euclidean distance + majority vote).
3. 解释缩放为何关键、K 的偏差方差权衡。
   Explain why scaling is critical and the bias–variance role of K.
4. 理解维度灾难对 KNN 的打击。
   Understand how the curse of dimensionality hurts KNN.
5. 了解加权投票与 KD-Tree 加速。
   Know weighted voting and KD-Tree speedups.

## 目录 / TOC
1. [先建直觉：物以类聚](#1)
2. [惰性学习与算法 ⭐](#2)
3. [🌸 数据 + 从零实现](#3)
4. [缩放为何关键 ⭐](#4)
5. [选 K：偏差方差 ⭐](#5)
6. [维度灾难 ⭐](#6)
7. [加权 KNN + 加速](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉：物以类聚 / Intuition First

KNN 的想法朴素到几乎不像算法：**要判断一个新点是什么类别，就看离它最近的 K 个邻居都是什么类别，少数服从多数。**
KNN's idea is almost too simple to be an algorithm: **to classify a new point, look at its K nearest neighbors and take the majority class.**

就像判断一个陌生人的口味，你会看他周围朋友的口味——"物以类聚"。这里没有要学习的"模型方程"，**决策完全由数据本身决定**。
Like guessing a stranger's taste from their friends' tastes — "birds of a feather". There is no "model equation" to learn; **the decision is dictated entirely by the data itself.**

代价是：因为不学习，每次预测都得把新点和**所有**训练点比一遍距离——数据一大就慢；而且它强烈依赖"距离"这个概念，于是缩放、维度都会深刻影响它（后面细讲）。
The price: since it doesn't learn, each prediction must compare the query against **all** training points — slow when data is big; and it leans heavily on the notion of "distance", so scaling and dimensionality deeply affect it (detailed below).


<a id="2"></a>
## 2. 惰性学习与算法 ⭐ / Lazy Learning & the Algorithm

预测一个点 $\mathbf{x}$ 的三步：
Three steps to predict a point $\mathbf{x}$:

1. 算 $\mathbf{x}$ 到**所有**训练点的距离（常用欧氏 $\|\mathbf{x}-\mathbf{x}_i\|_2$）。
   Compute the distance from $\mathbf{x}$ to **every** training point (usually Euclidean $\|\mathbf{x}-\mathbf{x}_i\|_2$).
2. 取距离最小的 **K** 个。
   Take the **K** closest.
3. **分类**：这 K 个里哪个类别最多就预测哪个（多数投票）；**回归**(4.10)：取它们的均值。
   **Classification**: predict the majority class among the K; **regression** (4.10): take their mean.

**复杂度**（面试高频）：
**Complexity** (frequently asked):
- **训练 Training**：$O(1)$——只是把数据存下来，不学任何参数，所以叫"惰性"。
  $O(1)$ — just stores the data, learns no parameters; hence "lazy".
- **预测 Prediction**：每个查询 $O(nd)$——要扫全部 $n$ 个点、每个点 $d$ 维。数据大就慢。
  $O(nd)$ per query — must scan all $n$ points across $d$ dimensions. Slow at scale.


<a id="3"></a>
## 3. 数据 + 从零实现 / Data + From Scratch

继续用 **Iris**（5.2 介绍过）：150 朵花，3 类，4 个特征。
Continuing with **Iris** (introduced in 5.2): 150 flowers, 3 classes, 4 features.

KNN 靠距离，所以**预测前必须缩放**（下一节解释），这里先标准化。
KNN relies on distance, so we **must scale before predicting** (next section explains); we standardize here first.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
sc = StandardScaler().fit(X_tr)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)

def knn_predict(X_train, y_train, X_query, k=5):
    preds = []
    for x in X_query:                                  # 对每个待预测的点逐个处理
        # X_train - x 用广播得到每个训练点与 x 的差向量; 平方后按行(axis=1)求和再开方 = 欧氏距离
        d = np.sqrt(((X_train - x) ** 2).sum(axis=1))  # d[i] = x 到第 i 个训练点的距离
        idx = np.argsort(d)[:k]                        # argsort 给出从近到远的下标, 取前 k 个 = 最近的 k 个邻居
        # y_train[idx] 是这 k 个邻居的标签; Counter.most_common(1) 取出现最多的那个标签(多数投票)
        preds.append(Counter(y_train[idx]).most_common(1)[0][0])
    return np.array(preds)

pred = knn_predict(Xtr, y_tr, Xte, k=5)
print(f"从零 KNN (k=5) test 准确率 from-scratch accuracy: {(pred == y_te).mean():.3f}")

from sklearn.neighbors import KNeighborsClassifier
sk = KNeighborsClassifier(n_neighbors=5).fit(Xtr, y_tr)
print(f"sklearn KNN (k=5) 准确率 accuracy: {sk.score(Xte, y_te):.3f} | 预测一致率 agreement {(pred==sk.predict(Xte)).mean():.3f}")


<a id="4"></a>
## 4. 缩放为何关键 ⭐ / Why Scaling Is Critical

KNN 完全靠**距离**。假如一个特征量纲很大（比如"收入"按元算，范围 0–100000），另一个很小（"年龄" 0–100），那么两点间的距离几乎**完全由大量纲特征决定**，小特征的差异被彻底淹没。
KNN lives on **distance**. If one feature has a large scale (e.g. "income" in dollars, range 0–100000) and another a small one ("age" 0–100), the distance between points is **dominated almost entirely by the large-scale feature**, drowning out the small one.

所以**用 KNN 前必须标准化**。下面故意把一个特征放大 1000 倍，看灾难，再看缩放如何救回来。
So we **must standardize before KNN**. Below we deliberately blow up one feature ×1000 to see the disaster, then how scaling rescues it.


In [ ]:
# 故意把第 0 个特征放大 1000 倍, 制造量纲悬殊 / blow up feature 0 by ×1000
X_bad = X_tr.copy(); X_bad[:, 0] *= 1000
X_bad_te = X_te.copy(); X_bad_te[:, 0] *= 1000

# 不缩放: 距离几乎只由被放大的那个特征决定 → 其它特征失效
acc_unscaled = KNeighborsClassifier(5).fit(X_bad, y_tr).score(X_bad_te, y_te)
# 先标准化(每列减均值除标准差)再训练/预测, 让各特征量纲一致
scaler = StandardScaler().fit(X_bad)
acc_scaled = KNeighborsClassifier(5).fit(scaler.transform(X_bad), y_tr).score(scaler.transform(X_bad_te), y_te)
print(f"某特征×1000 不缩放 unscaled: 准确率 {acc_unscaled:.3f}  (被该特征绑架 hijacked)")
print(f"标准化后 scaled:            准确率 {acc_scaled:.3f}  (恢复正常 back to normal)")
print("→ KNN 用前必须缩放; 距离对量纲极敏感 / always scale before KNN")


<a id="5"></a>
## 5. 选 K：偏差方差权衡 ⭐ / Choosing K: Bias–Variance

K 是 KNN 唯一的关键超参，它直接控制模型复杂度（接 3.10 偏差方差）：
K is KNN's one key hyperparameter, directly controlling model complexity (continuing the bias–variance idea, 3.10):

- **K 小（=1）**：决策边界**极度弯曲**，紧贴每个点 → **低偏差、高方差**，过拟合，对噪声敏感。
  **Small K (=1):** the boundary is **very jagged**, hugging each point → **low bias, high variance**, overfitting, noise-sensitive.
- **K 大**：边界**平滑**，趋向于"多数类" → **高偏差、低方差**；K=n 时退化成"永远预测最多的那一类"。
  **Large K:** the boundary is **smooth**, leaning toward the majority → **high bias, low variance**; at K=n it degenerates to "always predict the most common class".

通常用**交叉验证**选 K，且二分类时 K 取**奇数**避免平票。
Typically pick K by **cross-validation**, and for binary problems use an **odd** K to avoid ties.


In [ ]:
from sklearn.model_selection import cross_val_score
ks = range(1, 31)
# 对每个候选 K, 用 5 折交叉验证算平均准确率 (cross_val_score 返回每折分数, 取均值)
cv_acc = [cross_val_score(KNeighborsClassifier(k), Xtr, y_tr, cv=5).mean() for k in ks]
best_k = list(ks)[int(np.argmax(cv_acc))]   # argmax 找到 CV 准确率最高的那个 K

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(ks), cv_acc, "o-")
ax.axvline(best_k, color="r", ls="--", label=f"最佳 best K={best_k}")
ax.set_xlabel("K"); ax.set_ylabel("5 折 CV 准确率 5-fold CV accuracy")
ax.set_title("选 K: 太小过拟合(高方差), 太大欠拟合(高偏差) / small=overfit, large=underfit")
ax.legend(); plt.tight_layout(); plt.show()
print(f"CV 最佳 best K={best_k}, test 准确率 {KNeighborsClassifier(best_k).fit(Xtr,y_tr).score(Xte,y_te):.3f}")


In [ ]:
# 可视化 K=1 vs K=15 的决策边界 / decision boundary for K=1 vs K=15
X2 = Xtr[:, 2:4]                            # 取花瓣长/宽两维便于画图
# 生成网格点铺满整个平面, 用来给每个位置上色(显示该处会被分到哪一类)
xx, yy = np.meshgrid(np.linspace(X2[:,0].min()-.5, X2[:,0].max()+.5, 300),
                     np.linspace(X2[:,1].min()-.5, X2[:,1].max()+.5, 300))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, k in zip(axes, [1, 15]):
    m = KNeighborsClassifier(k).fit(X2, y_tr)
    # np.c_[xx.ravel(), yy.ravel()] 把网格摊平成两列坐标, 预测后 reshape 回网格形状
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")   # 背景色块 = 各区域的预测类别
    ax.scatter(X2[:,0], X2[:,1], c=y_tr, cmap="viridis", edgecolor="k", s=25)
    ax.set_title(f"K={k}: {'锯齿边界(高方差) jagged' if k==1 else '平滑边界(高偏差) smooth'}")
plt.tight_layout(); plt.show()


<a id="6"></a>
## 6. 维度灾难 ⭐ / Curse of Dimensionality

KNN 在**高维**会失灵：维度一高，**所有点彼此都差不多远**，"最近邻"就失去了意义。
KNN breaks down in **high dimensions**: as dimensionality grows, **all points become roughly equidistant**, so "nearest neighbor" loses meaning.

下面用实验验证：随机点在不同维度下，"最近距离 / 最远距离"的比值随维度升高而趋于 1（意味着最近和最远几乎一样远）。
The experiment below confirms it: for random points, the ratio "nearest distance / farthest distance" approaches 1 as dimensionality grows (nearest ≈ farthest).


In [ ]:
rng = np.random.default_rng(0)
dims = [2, 5, 10, 50, 100, 500]
ratios = []
for d in dims:
    P = rng.random((500, d))                       # 500 个随机点, 每个 d 维
    dist = np.sqrt(((P[0] - P[1:]) ** 2).sum(1))   # 第 0 个点到其余所有点的欧氏距离
    ratios.append(dist.min() / dist.max())         # 最近距离 / 最远距离
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dims, ratios, "o-")
ax.set_xlabel("维度 dimension d"); ax.set_ylabel("最近距离 / 最远距离 nearest/farthest")
ax.set_title("维度灾难: 高维下所有点几乎等距 → '最近邻'失效 / all points nearly equidistant")
plt.tight_layout(); plt.show()
print("距离比 ratios", [f"{r:.2f}" for r in ratios], "→ 维度越高越接近1, KNN 越失效")
print("应对 fix: 先降维(PCA, Part 6/7) 或只选少量有用特征 / reduce dimensions first")


<a id="7"></a>
## 7. 加权 KNN + 加速 / Weighted KNN & Speedups

- **加权投票 / Weighted voting**：让更近的邻居投票权更大（`weights='distance'`，权重 $\propto 1/d$），缓解 K 偏大时远邻的干扰。
  Give closer neighbors more voting power (`weights='distance'`, weight $\propto 1/d$), reducing the influence of far neighbors when K is large.
- **加速 / Speedups**：暴力预测是 $O(nd)$。**KD-Tree**（低维快）和 **Ball-Tree**（高维稍好）把空间预先组织成树，查询近似降到 $O(d\log n)$。sklearn 的 `algorithm='auto'` 会自动选。
  Brute-force prediction is $O(nd)$. **KD-Tree** (fast in low dim) and **Ball-Tree** (better in higher dim) pre-organize space into a tree, cutting queries to about $O(d\log n)$. sklearn's `algorithm='auto'` picks automatically.


In [ ]:
import time
print("均匀投票 vs 距离加权 / uniform vs distance weighting:")
for w in ["uniform", "distance"]:
    s = KNeighborsClassifier(15, weights=w).fit(Xtr, y_tr).score(Xte, y_te)
    print(f"  weights={w:<9} 准确率 accuracy {s:.3f}")

print("不同近邻搜索算法的预测耗时(结果相同, 仅速度不同) / search algorithms (same result, different speed):")
for algo in ["brute", "kd_tree", "ball_tree"]:
    m = KNeighborsClassifier(5, algorithm=algo).fit(Xtr, y_tr)
    t = time.perf_counter(); m.predict(Xte); dt = (time.perf_counter() - t) * 1000
    print(f"  algorithm={algo:<10} 预测 predict {dt:.2f} ms")


<a id="8"></a>
## 8. 小结 / Summary

```
KNN: 惰性非参数; 训练 O(1), 预测 O(nd); 多数投票(分类)/均值(回归 4.10)
必须缩放: 距离对量纲敏感, 大量纲特征会绑架结果
选 K: 小K=高方差(过拟合), 大K=高偏差(欠拟合); CV 选, 二分类取奇数
维度灾难: 高维下点近乎等距, "最近邻"失效 → 先降维(PCA)
加权投票(1/d) + KD/Ball-Tree 加速
```

### 💡 面试速查 / Interview cheat-sheet
1. **训练 O(1)、预测 O(nd)** —— 惰性，代价全在预测。
   O(1) training, O(nd) prediction — lazy, all cost at predict time.
2. **必须缩放** —— 距离对量纲敏感。
   Must scale — distance is scale-sensitive.
3. **K 小过拟合 / K 大欠拟合** —— CV 选 K。
   Small K overfits, large K underfits — choose K by CV.
4. **维度灾难** —— 高维下 KNN 失效，先 PCA 降维。
   Curse of dimensionality — KNN fails in high dim; reduce with PCA first.
5. 加速用 **KD-Tree（低维）/ Ball-Tree（较高维）**。
   Speed up with KD-Tree (low dim) / Ball-Tree (higher dim).

### 下一节 / Next
**5.4 朴素贝叶斯**——又一个非参数视角：用贝叶斯定理(2.8)直接建模 P(类|特征)，"朴素"地假设特征条件独立，是文本分类的经典快枪手。
**5.4 Naive Bayes** — another angle: model P(class|features) via Bayes' theorem (2.8), with the "naive" conditional-independence assumption; a classic fast baseline for text.
